In [1]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/edimaudo/LibraryTO.git"
BUILDER = Path("scripts") / "build_relate_graph.py"

# Locate an existing LibraryTO checkout.
search_locations = [
    Path.cwd(),
    *Path.cwd().parents,
    Path("/content/LibraryTO"),
    Path("/content/drive/MyDrive/LibraryTO"),
]

repo_root = None
checked = set()

for location in search_locations:
    location = location.resolve()

    if location in checked:
        continue

    checked.add(location)

    if (location / BUILDER).is_file():
        repo_root = location
        break

# Clone the repository if it is not already available.
if repo_root is None:
    repo_root = Path("/content/LibraryTO")

    # Avoid overwriting an existing directory.
    suffix = 1
    while repo_root.exists():
        repo_root = Path(f"/content/LibraryTO_colab_{suffix}")
        suffix += 1

    print("Cloning LibraryTO repository...")

    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
        check=True
    )

# Confirm that the graph builder exists.
builder_path = repo_root / BUILDER

if not builder_path.is_file():
    raise FileNotFoundError(
        f"Graph builder not found: {builder_path}"
    )

# Verify the seven required CSV files.
required_csvs = [
    "tpl-events-feed.csv",
    "tpl-Neighbourhoods.csv",
    "tpl-circulation-annual-by-branch.csv",
    "tpl-branch-general-information.csv",
    "tpl-workstation-usage-annual-by-branch.csv",
    "tpl-visits-annual-by-branch.csv",
    "tpl-card-registrations-annual-by-branch.csv",
]

data_dir = repo_root / "data"

missing_files = [
    filename for filename in required_csvs
    if not (data_dir / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required CSV files:\n" +
        "\n".join(missing_files)
    )

# Install the project's pinned Relate database dependency.
print("Installing Relate dependency...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "relate-db==1.1.0"
    ],
    check=True
)

# Run the graph builder from the repository.
print(f"\nBuilding Relate graph from: {repo_root}")

subprocess.run(
    [sys.executable, str(builder_path)],
    cwd=str(repo_root),
    check=True
)

graph_file = data_dir / "tpl_library_graph.rl"

if not graph_file.is_file():
    raise FileNotFoundError("Graph build completed without producing the expected database.")

print("\nRelate graph successfully rebuilt.")
print(f"Graph location: {graph_file}")
print(f"Graph size: {graph_file.stat().st_size / (1024 * 1024):.2f} MB")

Cloning LibraryTO repository...
Installing Relate dependency...

Building Relate graph from: /content/LibraryTO

Relate graph successfully rebuilt.
Graph location: /content/LibraryTO/data/tpl_library_graph.rl
Graph size: 12.35 MB
